# Prefix robustness audit
## Question and why it matters
Does the W3c prefix-8 result survive cross-fitting and nonlinear area adjustment?
This audits the surviving rule-space lead; it does not identify a mechanism.
See [registration and results](../README.md) and [research questions](../../../../notes/open-questions-hypotheses.md).
## Assumptions and definitions
Stationary means |g_k| < 0.05; rules need six core-band runs. Rule means are the units.
Cross-fitting holds out rules, allowing shared prefixes; this tests new suffixes of known prefixes.
Only ln A(22) is controlled, not the full growth history. This assumption limits the interpretation.


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Cargo.toml').exists())
EXP = ROOT / 'experiments/archive/2026-10-kappa-rule-space'
paths = [EXP / 'results/prefix_audit.json', EXP / 'results/prefix_audit_predictions.csv']
for p in paths:
    assert p.exists(), p
    print(p.relative_to(ROOT), hashlib.sha256(p.read_bytes()).hexdigest())
report = json.loads(paths[0].read_text())
SEED = report['seed']
np.random.seed(SEED)
print('git', report['git'], 'seed', SEED)
print('analysis input/source hashes:', json.dumps(report['sha256'], indent=2))
print(report['calibration'])


experiments/2026-10-kappa-rule-space/results/prefix_audit.json 348d7087665c5db1a155f9e2d29f4282f3204bf19f46d717ed4f7179b3b8cc14
experiments/2026-10-kappa-rule-space/results/prefix_audit_predictions.csv 742f0fe1d776696ea7d0bbd8e889128d6cc979d912e13736801a22165023dcd2
git 275c2afe3e03b9d08600447acf0840e5ca3d9543 seed 20261015
analysis input/source hashes: {
  "experiments/2026-10-kappa-rule-space/dumps/w3c_probe.csv": "28ddbc4f61e98b50ddb0f79b161b92a41d7e8593cfcc31a8a474f0b25f22027a",
  "experiments/2026-10-13-probe-growth-census/results/probe_summary.csv": "c76c7927185b1cee273433c340cfa393fb16696fbe4a6f155fc49c33ef32cb5d",
  "experiments/2026-10-13-probe-growth-census/results/probe_all.csv": "b1f1c4a2d87b82c1b5cddcb73d0100c8ea1233e55dcb2e83b54625348ab43d15",
  "experiments/2026-10-kappa-rule-space/scripts/prefix_audit.py": "861297a24ed8b90dc8e60b9d575bdee7b518fb0f08137fab3dedc66450a88df0",
  "experiments/2026-10-kappa-rule-space/scripts/kcommon.py": "fd21bb37cb23306dff0677b443f08ab22951

## Data and method
The producing script is scripts/prefix_audit.py; raw trajectories stay outside this notebook.
Five-fold training lookup or historical LOO lookup; final lookup uses full census training.
OLS compares area alone to area plus lookup. Flexible area control has five training-quantile hinge knots.
Intervals resample 2000 test-rule sets, conditioning on training. Model choices were frozen before this audit.
Results were uncommitted when prepared; source and input hashes are recorded. Repository policy defers commits to user request.
## Results


In [2]:
print('cohort     encoding   area    N   baseline R2  model R2  gain   CI             without top5')
for r in report['rows']:
    print(f"{r['cohort']:10} {r['encoding']:10} {r['area']:6} {r['n_test']:3} {r['baseline_r2']:11.3f} {r['model_r2']:9.3f} {r['gain']:6.3f} [{r['ci'][0]:.3f},{r['ci'][1]:.3f}] {r['without_top5']:.3f}")
print('Frozen expectations:', report['expectations'])
print('All prefix8 support:', all(r['prefix8_share'] == 1 for r in report['rows']))
print('Test areas outside training range:', [r['area_outside_training'] for r in report['rows']])


cohort     encoding   area    N   baseline R2  model R2  gain   CI             without top5
stationary loo        linear 162       0.497     0.737  0.240 [0.153,0.344] 0.197
stationary loo        spline 162       0.528     0.727  0.198 [0.099,0.311] 0.137
stationary crossfit5  linear 162       0.497     0.736  0.240 [0.156,0.342] 0.198
stationary crossfit5  spline 162       0.528     0.729  0.201 [0.103,0.311] 0.150
all        loo        linear 176       0.363     0.394  0.031 [-0.000,0.063] 0.009
all        loo        spline 176       0.472     0.512  0.040 [-0.030,0.112] -0.013
all        crossfit5  linear 176       0.363     0.389  0.026 [0.001,0.052] 0.008
all        crossfit5  spline 176       0.472     0.514  0.041 [-0.018,0.103] -0.003
Frozen expectations: {'A1': True, 'A2': True}
All prefix8 support: False
Test areas outside training range: [0, 0, 0, 0, 0, 0, 0, 0]


The stationary gain survives cross-fitting and a flexible area baseline; the all-rules estimate remains weak.
This does not establish a transport mechanism, asymptotic law, or independence from growth history.
## Caveats and controls
Known area-only and planted-signal calibration passes; original W3c point estimate is reproduced.
The existing letters-to-c positive control passed in W5; this audit does not repeat it or a label-permutation test.
No new trajectories, stationarity calibration, or noise-ceiling estimate were made here.
Bootstrap rules share prefixes, so the intervals describe rule resampling rather than transfer to unseen prefix families.
The five-largest-contributor removal is a diagnostic selected from test outcomes.
Both the test cohort and the prefix were previously inspected: this is robustness, not fresh confirmation.
## What this changes / next steps
Retain the stationary prefix lead. A causal next experiment should compare mutations at fixed area,
and distinguish early-colour exposure from changes to the entire self-organised field.
Open questions: why stationary training is essential, whether area measurement noise leaves a residual size effect,
and whether prefix influence persists in longer-time, matched-area mutation runs.
